# 隐藏层

我们成功训练了第一个神经网络模型，它可以根据天气预报预测冰激凌的销量。

但冰激凌的销量真的是由天气直接决定的吗？

事实上，天气影响的是人们的**行为和偏好**，例如是否愿意出门、是否想吃冰激凌；而这些行为和偏好，才是真正影响销量的因素：

* 天气凉爽时：人们更愿意出门，但未必想吃冰激凌；
* 天气酷热时：人们更想吃冰激凌，但可能不愿意出门；
* 天气寒冷时：人们既不愿意出门，也不想吃冰激凌。

---

这启发了一个新的建模思路：

* 先用一个模型，根据天气预测这些**中间因素**（出门意愿、吃冰激凌意愿）；
* 再用另一个模型，根据中间因素预测最终的冰激凌销量。

更进一步：把前一个模型的输出直接作为后一个模型的输入，将两者连接成一个整体。训练时，只需要销量的预测误差，通过**反向传播**逐层传递，就可以同时训练两个模型。

这种在网络内部自动学习中间因素的能力，正是深度神经网络强大之处的来源。

## 层

我们把模型分成两**层**（Layer）：

* **第一层**：根据天气预测中间值（如出门意愿、吃冰激凌意愿）；
* **第二层**：根据中间值预测最终销量。

这样的两层模型是否可行？能训练成功吗？本章将一探究竟。

In [1]:
import numpy as np

np.random.seed(99)

``💡 np.random.seed 固定随机数种子，确保每次运行生成相同的随机初始参数，结果可复现。``

## 数据集

与前几章相同，直接沿用。

### 训练数据：特征、标签

In [2]:
train_features = np.array([[22.5, 72.0],
                           [31.4, 45.0],
                           [19.8, 85.0],
                           [27.6, 63.0]])
train_labels = np.array([[95],
                         [210],
                         [70],
                         [155]])

### 测试数据：特征、标签

In [3]:
test_features = np.array([[28.1, 58.0]])
test_labels = np.array([[165]])

## 模型

两层网络的数据流形成三条传播链路：

1. **前向传播链路**

```{figure} images/layer-forward.png
:align: center
:width: 480px

* $x$：特征值；
* $w_1, b_1$：第一层权重和偏置；
* $h$：中间值；
* $w_2, b_2$：第二层权重和偏置；
* $p$：预测值。

```

2. **梯度计算链路**

```{figure} images/layer-gradient.png
:align: center
:width: 460px

* $p$：预测值；
* $y$：标签值；
* $d_2$：第二层误差项；
* $w_2$：第二层权重；
* $d_1$：第一层误差项。

```

3. **反向传播链路**

```{figure} images/layer-backward.png
:align: center
:width: 240px

* $d_2$：第二层误差项；
* $h$：中间值；
* $w_2, b_2$：第二层权重和偏置；
* $d_1$：第一层误差项；
* $x$：特征值；
* $w_1, b_1$：第一层权重和偏置。

```

---

两层网络的推理公式是：

$$
h = w_1 \cdot x + b_1 \qquad p = w_2 \cdot h + b_2
$$

代入后得：

$$
p = w_2 \cdot (w_1 \cdot x + b_1) + b_2 = \underbrace{(w_2 \cdot w_1)}_{\text{等价权重}} \cdot x + \underbrace{(w_2 \cdot b_1 + b_2)}_{\text{等价偏置}}
$$

这个公式揭示了一个重要事实：**两层线性网络在数学上等价于一层线性网络**，只不过权重和偏置换了一个等价的形式。不论堆叠多少层线性变换，最终都可以折叠成一层。这意味着仅靠增加线性层，并不能增加模型的表达能力。

``💡 既然如此，为什么还要研究多层网络？因为解决这个问题的方案：在层与层之间引入非线性激活函数，才是深度神经网络真正强大的关键。``

### 参数：隐藏层权重、偏置

实践中我们不需要关心中间值的实际含义，它们只是给下一层提供输入。因此第一层通常被称为**隐藏层**（Hidden Layer），泛指所有输出中间值的层。

我们的隐藏层将有 4 个神经元，每个神经元对应一组权重（作用于 2 个输入特征值），因此权重矩阵形状为 `(4, 2)`，偏置形状为 `(4,)`（每个神经元一个偏置）。

**为什么改用随机数初始化，而不是常量？**

如果所有神经元初始权重完全相同，反向传播后每个神经元收到的梯度也完全相同，导致参数始终保持一致。4 个神经元永远等价于 1 个神经元。随机初始化**打破了这种对称性**，使每个神经元从不同起点出发，最终学到不同的特征。

In [4]:
hidden_weight = np.random.rand(4, 2) / 2  # shape (4, 2)：4个神经元，每个作用于2个特征
hidden_bias = np.random.rand(4)  # shape (4,) ：每个神经元一个偏置

### 参数：输出层权重、偏置

最后一层称为**输出层**（Output Layer），其输出是最终预测值。

输出层接收隐藏层的 4 个中间值，输出 1 个预测值，因此权重形状为 `(1, 4)`，偏置形状为 `(1,)`。

In [5]:
output_weight = np.random.rand(1, 4) / 4  # shape (1, 4)：1个输出神经元，接收4个隐藏值
output_bias = np.random.rand(1)  # shape (1,) ：1个偏置

### 推理函数

两层网络共用同一个推理函数，分别调用两次：第一次计算隐藏值 $h$，第二次计算预测值 $p$。

In [6]:
def forward(x, w, b):
    return x @ w.T + b

### 损失函数（均方误差）

In [7]:
def mse_loss(p, y):
    return np.mean(np.square(y - p))

### 梯度函数

In [8]:
def gradient(p, y):
    return -2 * (y - p) / len(y)

### 梯度反向函数

这是本章新增的函数，用于将**输出层的误差项**反向传播给**隐藏层**。

根据链式规则，隐藏层的误差项为：

$$
d_1 = \frac{\partial L}{\partial h} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial h} = d_2 \cdot w_2
$$

其中 $\frac{\partial p}{\partial h} = w_2$，是输出层推理函数关于中间值 $h$ 的导数。

维度分析：
* $d_2$：`(批大小, 1)`
* $w_2$：`(1, 4)`
* $d_1 = d_2 \cdot w_2$：`(批大小, 1) @ (1, 4)` = `(批大小, 4)`

结果是每个样本、每个隐藏神经元各有一个误差项，形状与隐藏层的输出一致。

In [9]:
def gradient_backward(d, w):
    return d @ w

### 反向函数

与前几章相同，通用于两层的参数更新。

In [10]:
def backward(x, d, w, b, lr):
    w -= d.T @ x * lr
    b -= np.sum(d, axis=0) * lr
    return w, b

## 训练

### 超参数：学习率

In [11]:
LEARNING_RATE = 0.00001

### 超参数：批大小

In [12]:
BATCH_SIZE = 2

### 超参数：轮数

In [13]:
EPOCHS = 1000

### 迭代

两层网络的训练循环与单层相同，只是前向传播、梯度计算、反向传播各自执行两次：隐藏层和输出层各一次。

注意梯度计算的顺序：必须**先计算输出层误差项**，再用更新前输出层权重**计算隐藏层误差项**（代码中 `gradient_backward()` 在 `backward()` 之前调用，保证了这一顺序）。

In [14]:
for epoch in range(EPOCHS):
    for i in range(0, len(train_features), BATCH_SIZE):
        features = train_features[i: i + BATCH_SIZE]
        labels = train_labels[i: i + BATCH_SIZE]

        # 前向传播（两步）
        hidden = forward(features, hidden_weight, hidden_bias)
        predictions = forward(hidden, output_weight, output_bias)

        # 梯度计算：从输出层逐层向隐藏层传播误差
        output_delta = gradient(predictions, labels)  # 输出层误差项
        hidden_delta = gradient_backward(output_delta, output_weight)  # 隐藏层误差项

        # 反向传播：先更新输出层，再更新隐藏层
        output_weight, output_bias = backward(hidden, output_delta, output_weight, output_bias, LEARNING_RATE)
        hidden_weight, hidden_bias = backward(features, hidden_delta, hidden_weight, hidden_bias, LEARNING_RATE)

print(f'hidden weight:\n{hidden_weight}')
print(f'output weight: {output_weight}')

hidden weight:
[[ 1.45353425 -0.14661746]
 [ 1.38767435 -0.33933946]
 [ 2.0254864  -0.21125674]
 [ 0.69613316 -0.14191958]]
output weight: [[1.35796276 1.30704936 1.91057461 0.67564236]]


## 验证

### 推理

推理时同样需要经过两层前向传播。

In [15]:
hidden = forward(test_features, hidden_weight, hidden_bias)
predictions = forward(hidden, output_weight, output_bias)
print(f'predictions: {predictions}')

predictions: [[166.58568956]]


### 评估

In [16]:
loss = mse_loss(predictions, test_labels)
print(f'loss: {loss:.4f}')

loss: 2.5144


两层网络的最终损失 `2.51` 与上一章单层网络 `2.18` 几乎相同。

这个结果正是本章开头数学推导所预言的：**两层线性网络等价于单层线性网络**。不论把它们训练多少轮、调整多少参数，本质上仍然是在拟合同一类线性函数。增加线性层的数量，并没有增加模型的表达能力。

## 课后练习

将隐藏层神经元数从 4 改为 2 或 8，观察最终损失值是否有变化？结合本章的数学推导，解释为什么改变神经元数量对线性网络的最终效果没有实质影响。